In [1]:
import bayesflow as bf
import torch
import numpy as np
import pandas as pd
import keras
import os
import matplotlib.pyplot as plt

from DDM_DC_Pedestrain import prior_DC, ddm_DC_alphaToCpp, all_models, CONDITIONS, meta, adopt

ModuleNotFoundError: No module named 'bayesflow'

In [ ]:
# Check available models and conditions
print("Available models:", list(all_models.keys()))
print("TTA conditions:", CONDITIONS)
print("Parameter names:", list(prior_DC().keys()))

## 1. Understanding the Conditional Structure

Let's first understand how the simulator generates data with conditions:

In [ ]:
# Get the model and adapter
model_name = 'model_DC'
model, adapter = all_models[model_name]

# Generate some sample data to understand the structure
print("Generating 3 simulations to understand data structure...\n")
sample_data = model.sample(3)

print("Keys in simulated data:", sample_data.keys())
print("\nShape of 'x' (observed data):", sample_data['x'].shape)
print("  -> (n_simulations=3, n_trials=60, 2 features: [RT, CPP])")
print("\nTTA condition for each simulation:", sample_data['tta_condition'])
print("\nExample parameters (first simulation):")
for key in ['theta', 'b0', 'k', 'mu_ndt']:
    if key in sample_data:
        print(f"  {key}: {sample_data[key][0]:.4f}")

## 2. Load Trained Model

If you've already trained a model, load it here:

In [ ]:
base_dir = os.getcwd()
save_dir = os.path.join(base_dir, "trained_model1", "checkpoints")
save_path = os.path.join(save_dir, model_name + ".keras")

if os.path.exists(save_path):
    print(f"Loading model from: {save_path}")
    approximator = keras.saving.load_model(save_path)
    print("✓ Model loaded successfully!")
else:
    print(f"⚠ No trained model found at {save_path}")
    print("You need to train the model first using train.py or main.py")
    approximator = None

## 3. Parameter Recovery: Overall Performance

First, let's check overall parameter recovery across all TTA conditions:

In [ ]:
if approximator is not None:
    # Generate validation data (mixture of all TTA conditions)
    num_samples = 1000  # Posterior samples per simulation
    n_validation = 200   # Number of validation simulations
    
    print(f"Generating {n_validation} validation simulations...")
    val_sims = model.sample(n_validation)
    
    # Get posterior samples
    print(f"Drawing {num_samples} posterior samples per validation simulation...")
    post_draws = approximator.sample(conditions=val_sims, num_samples=num_samples)
    
    # Plot recovery
    par_names = ['theta', 'b0', 'k', 'mu_ndt', 'sigma_ndt', 'mu_alpah', 'sigma_alpha', 'sigma_cpp']
    
    f = bf.diagnostics.plots.recovery(
        estimates=post_draws,
        targets=val_sims,
        variable_names=par_names
    )
    plt.suptitle("Parameter Recovery: All TTA Conditions Combined", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Cannot run evaluation without a trained model")

## 4. Condition-Specific Parameter Recovery

Now let's evaluate parameter recovery for each TTA condition separately:

In [ ]:
def filter_by_condition(data_dict, tta_value, tolerance=0.01):
    """
    Filter simulations by TTA condition.
    
    Args:
        data_dict: Dictionary from model.sample() containing 'tta_condition'
        tta_value: Target TTA value (e.g., 2.5, 3.0, 3.5, 4.0)
        tolerance: Numerical tolerance for matching
    
    Returns:
        Filtered dictionary with only matching simulations
    """
    tta_conditions = data_dict['tta_condition']
    mask = np.abs(tta_conditions - tta_value) < tolerance
    
    filtered = {}
    for key, value in data_dict.items():
        if isinstance(value, np.ndarray) and len(value) == len(tta_conditions):
            filtered[key] = value[mask]
        else:
            filtered[key] = value
    
    return filtered

In [ ]:
if approximator is not None:
    # Generate more data to ensure enough samples per condition
    n_per_condition = 50
    total_needed = n_per_condition * len(CONDITIONS) * 2  # oversample
    
    print(f"Generating {total_needed} simulations to get ~{n_per_condition} per condition...")
    val_sims_large = model.sample(total_needed)
    
    par_names = ['theta', 'b0', 'k', 'mu_ndt', 'sigma_ndt', 'mu_alpah', 'sigma_alpha', 'sigma_cpp']
    
    # Evaluate each condition separately
    for tta in CONDITIONS:
        print(f"\n{'='*60}")
        print(f"Evaluating TTA = {tta}s")
        print(f"{'='*60}")
        
        # Filter data for this condition
        cond_data = filter_by_condition(val_sims_large, tta)
        n_sims = len(cond_data['tta_condition'])
        print(f"Found {n_sims} simulations for TTA={tta}s")
        
        if n_sims < 10:
            print(f"  ⚠ Too few samples, skipping...")
            continue
        
        # Get posterior samples for this condition
        post_draws_cond = approximator.sample(conditions=cond_data, num_samples=500)
        
        # Plot recovery for this condition
        f = bf.diagnostics.plots.recovery(
            estimates=post_draws_cond,
            targets=cond_data,
            variable_names=par_names
        )
        plt.suptitle(f"Parameter Recovery for TTA = {tta}s", y=1.02, fontsize=14)
        plt.tight_layout()
        plt.show()
else:
    print("Cannot run evaluation without a trained model")

## 5. Applying to Real Experimental Data

Here's how to prepare and analyze real data from your pedestrian experiment.

### Data Structure Expected:
For each subject, you have 256 trials:
- 64 trials at TTA = 2.5s
- 64 trials at TTA = 3.0s  
- 64 trials at TTA = 3.5s
- 64 trials at TTA = 4.0s

Each trial has:
- Reaction time (RT) - when they pressed the key
- CPP measure (if available from EEG)
- TTA condition

In [ ]:
def prepare_real_data_for_inference(subject_data_df, tta_column='TTA', rt_column='RT', cpp_column='CPP'):
    """
    Prepare real experimental data for BayesFlow inference.
    
    Args:
        subject_data_df: DataFrame with columns for TTA, RT, and CPP
        tta_column: Name of TTA column
        rt_column: Name of reaction time column  
        cpp_column: Name of CPP column (use None if CPP not available)
    
    Returns:
        Dictionary in BayesFlow format ready for inference
    """
    # Group by TTA condition
    unique_ttas = np.sort(subject_data_df[tta_column].unique())
    
    data_by_condition = {}
    
    for tta in unique_ttas:
        condition_trials = subject_data_df[subject_data_df[tta_column] == tta]
        
        # Extract RT
        rts = condition_trials[rt_column].values
        
        # Extract or simulate CPP
        if cpp_column is not None and cpp_column in condition_trials.columns:
            cpps = condition_trials[cpp_column].values
        else:
            # If no CPP available, use placeholder (zeros or NaN)
            # Note: Model performance may degrade without CPP
            cpps = np.zeros_like(rts)
            print(f"⚠ Warning: No CPP data found, using zeros")
        
        # Stack into [n_trials, 2] format
        x = np.column_stack([rts, cpps])
        
        data_by_condition[tta] = x
    
    return data_by_condition


def infer_subject_parameters(approximator, adapter, data_by_condition, num_samples=2000):
    """
    Infer subject-level parameters using all TTA conditions.
    
    Args:
        approximator: Trained BayesFlow approximator
        adapter: The adapter function
        data_by_condition: Dict mapping TTA -> trial data array
        num_samples: Number of posterior samples to draw
    
    Returns:
        Dictionary with posterior samples for each parameter
    """
    all_posteriors = []
    
    for tta, trials in data_by_condition.items():
        # Prepare data in BayesFlow format
        data_dict = {
            'x': trials[np.newaxis, :, :],  # Add batch dimension: [1, n_trials, 2]
            'tta_condition': np.array([tta]),  # TTA for this condition
            'number_of_trials': np.array([len(trials)])
        }
        
        # Apply adapter transformations
        adapted_data = adapter(data_dict)
        
        # Get posterior samples
        posterior = approximator.sample(conditions=adapted_data, num_samples=num_samples)
        all_posteriors.append(posterior)
        
        print(f"  TTA={tta}s: {len(trials)} trials -> {num_samples} posterior samples")
    
    # Combine posteriors from all conditions
    # Average across conditions (you might want to concatenate instead)
    combined_posterior = {}
    param_names = all_posteriors[0].keys()
    
    for param in param_names:
        # Stack all condition-specific posteriors and average
        # Shape: [n_conditions, num_samples, 1] -> [num_samples,]
        stacked = np.concatenate([p[param] for p in all_posteriors], axis=0)
        combined_posterior[param] = stacked
    
    return combined_posterior

### Example: Simulated Real Data Analysis

In [ ]:
if approximator is not None:
    # Simulate "real" data for demonstration
    # In practice, you would load your actual experimental data here
    
    print("Simulating 'real' subject data...\n")
    
    # Generate ground truth parameters for a simulated subject
    true_params = prior_DC()
    print("True parameters for simulated subject:")
    for param, value in true_params.items():
        print(f"  {param}: {value:.4f}")
    
    # Simulate data for each TTA condition (64 trials each)
    simulated_subject_data = []
    
    for tta in CONDITIONS:
        # Simulate 64 trials for this TTA condition
        data = ddm_DC_alphaToCpp(
            theta=true_params['theta'],
            b0=true_params['b0'],
            k=true_params['k'],
            mu_ndt=true_params['mu_ndt'],
            sigma_ndt=true_params['sigma_ndt'],
            mu_alpah=true_params['mu_alpah'],
            sigma_alpha=true_params['sigma_alpha'],
            sigma_cpp=true_params['sigma_cpp'],
            number_of_trials=64,
            tta_condition=tta
        )
        
        trials = data['x']  # [64, 2] array
        
        for trial_idx in range(len(trials)):
            simulated_subject_data.append({
                'TTA': tta,
                'RT': trials[trial_idx, 0],
                'CPP': trials[trial_idx, 1],
                'trial_idx': trial_idx
            })
    
    # Convert to DataFrame (mimicking real data structure)
    subject_df = pd.DataFrame(simulated_subject_data)
    
    print(f"\nSimulated subject data: {len(subject_df)} total trials")
    print(subject_df.groupby('TTA').size())
    print("\nFirst few rows:")
    print(subject_df.head(10))
else:
    print("Cannot run evaluation without a trained model")

In [ ]:
if approximator is not None:
    # Prepare data for inference
    print("\nPreparing data for inference...")
    data_by_condition = prepare_real_data_for_inference(subject_df)
    
    print(f"\nData prepared for {len(data_by_condition)} TTA conditions:")
    for tta, trials in data_by_condition.items():
        print(f"  TTA={tta}s: {len(trials)} trials")
    
    # Infer parameters
    print("\nRunning inference...")
    posterior_samples = infer_subject_parameters(
        approximator=approximator,
        adapter=adapter,
        data_by_condition=data_by_condition,
        num_samples=2000
    )
    
    print("\n" + "="*60)
    print("INFERENCE RESULTS")
    print("="*60)
    
    # Compare inferred vs true parameters
    param_names = ['theta', 'b0', 'k', 'mu_ndt', 'sigma_ndt', 'mu_alpah', 'sigma_alpha', 'sigma_cpp']
    
    results_comparison = []
    
    for param in param_names:
        post_samples = posterior_samples[param].flatten()
        
        # Summary statistics
        mean_est = np.mean(post_samples)
        median_est = np.median(post_samples)
        std_est = np.std(post_samples)
        ci_lower = np.percentile(post_samples, 2.5)
        ci_upper = np.percentile(post_samples, 97.5)
        
        true_val = true_params[param]
        
        results_comparison.append({
            'Parameter': param,
            'True': f"{true_val:.4f}",
            'Mean': f"{mean_est:.4f}",
            'Median': f"{median_est:.4f}",
            '95% CI': f"[{ci_lower:.4f}, {ci_upper:.4f}]",
            'In CI': '✓' if ci_lower <= true_val <= ci_upper else '✗'
        })
    
    results_df = pd.DataFrame(results_comparison)
    print(results_df.to_string(index=False))
    
    # Plot posterior distributions
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx, param in enumerate(param_names):
        ax = axes[idx]
        post_samples = posterior_samples[param].flatten()
        
        ax.hist(post_samples, bins=50, density=True, alpha=0.6, color='blue', edgecolor='black')
        ax.axvline(true_params[param], color='red', linestyle='--', linewidth=2, label='True')
        ax.axvline(np.mean(post_samples), color='green', linestyle='-', linewidth=2, label='Posterior Mean')
        ax.set_xlabel(param)
        ax.set_ylabel('Density')
        ax.legend()
        ax.set_title(f"{param}")
    
    plt.suptitle("Posterior Distributions vs True Parameters", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Cannot run evaluation without a trained model")

## 6. Loading Your Actual Data

To use your real experimental data, uncomment and modify the code below:

In [ ]:
# # Load your actual data
# # Adjust the path and format based on your data structure
# 
# real_data_path = "path/to/your/data.csv"
# real_df = pd.read_csv(real_data_path)
# 
# # Expected columns: 'subject_id', 'TTA', 'RT', 'CPP' (if available)
# # You may need to preprocess:
# # - Convert RT to seconds
# # - Handle cross-after decisions (RT = -1 or special code)
# # - Filter out practice trials
# 
# # For each subject:
# unique_subjects = real_df['subject_id'].unique()
# 
# all_subject_results = []
# 
# for subject_id in unique_subjects:
#     print(f"\nProcessing subject {subject_id}...")
#     
#     subject_data = real_df[real_df['subject_id'] == subject_id]
#     
#     # Prepare data
#     data_by_condition = prepare_real_data_for_inference(
#         subject_data,
#         tta_column='TTA',
#         rt_column='RT',
#         cpp_column='CPP'  # or None if not available
#     )
#     
#     # Infer parameters
#     posterior = infer_subject_parameters(
#         approximator=approximator,
#         adapter=adapter,
#         data_by_condition=data_by_condition,
#         num_samples=2000
#     )
#     
#     # Store results
#     subject_result = {'subject_id': subject_id}
#     for param in param_names:
#         subject_result[f"{param}_mean"] = np.mean(posterior[param])
#         subject_result[f"{param}_std"] = np.std(posterior[param])
#     
#     all_subject_results.append(subject_result)
# 
# # Compile results across subjects
# results_all_subjects = pd.DataFrame(all_subject_results)
# print("\nResults across all subjects:")
# print(results_all_subjects.head())
# 
# # Save results
# results_all_subjects.to_csv("inferred_parameters_all_subjects.csv", index=False)
# print("\n✓ Results saved to 'inferred_parameters_all_subjects.csv'")

## Summary

This notebook showed how to:

1. **Train with conditions**: The model learns $p(\theta | \text{data}, \text{TTA})$
2. **Evaluate by condition**: Check parameter recovery for each TTA separately
3. **Apply to real data**: Prepare experimental data and infer subject-level parameters

### Key Points:

- Each simulation during training has ONE TTA condition (randomly selected)
- The network learns how TTA affects the observed data
- For real data, you provide trials grouped by TTA condition
- The network uses the TTA information when inferring parameters

### Next Steps:

1. Train the model with more epochs (e.g., 50-100 epochs)
2. Verify parameter recovery is good across all conditions
3. Apply to your real experimental data
4. Analyze differences in parameters across subjects or experimental manipulations